# Lab 13: Frequent Itemset Mining and Association Rule Generation

**Aim:** Discover frequent itemsets and generate association rules using the Apriori algorithm.

Import the libraries required for encoding transactions and mining rules.

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

Create a small sample dataset of customer transactions.

In [ ]:
transactions = [
    ['Milk', 'Bread', 'Butter'],
    ['Bread', 'Butter'],
    ['Milk', 'Bread'],
    ['Milk', 'Butter'],
    ['Milk', 'Bread', 'Butter', 'Eggs']
]

for i, transaction in enumerate(transactions, start=1):
    print(f"T{i}: {transaction}")

Convert the transactions into one-hot encoded columns (1 = item present, 0 = absent).

In [ ]:
encoder = TransactionEncoder()
encoded_array = encoder.fit(transactions).transform(transactions)
encoded_df = pd.DataFrame(encoded_array, columns=encoder.columns_)

display(encoded_df)

Use Apriori to find itemsets that appear in at least 30% of transactions.

In [ ]:
frequent_itemsets = apriori(encoded_df, min_support=0.3, use_colnames=True)

frequent_itemsets['itemsets'] = frequent_itemsets['itemsets'].apply(
    lambda items: ', '.join(sorted(items))
)
display(frequent_itemsets)

Generate rules with a minimum confidence of 0.5 and display their evaluation measures.

In [ ]:
# Recalculate itemsets with frozenset values for association_rules
frequent_itemsets_for_rules = apriori(encoded_df, min_support=0.3, use_colnames=True)
rules = association_rules(
    frequent_itemsets_for_rules,
    metric="confidence",
    min_threshold=0.5
)

if rules.empty:
    print("No association rules were generated.")
else:
    rules_display = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
    rules_display['antecedents'] = rules_display['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    rules_display['consequents'] = rules_display['consequents'].apply(lambda x: ', '.join(sorted(x)))
    display(rules_display.round(3))

Find the rule with the highest lift and review the three main measures.

In [ ]:
if not rules.empty:
    best_rule = rules.loc[rules['lift'].idxmax()]
    print("Rule with highest lift:")
    print(f"{', '.join(sorted(best_rule['antecedents']))} -> {', '.join(sorted(best_rule['consequents']))}")
    print(f"Support: {best_rule['support']:.3f}")
    print(f"Confidence: {best_rule['confidence']:.3f}")
    print(f"Lift: {best_rule['lift']:.3f}")
else:
    print("There are no rules to compare.")

print("\nSupport: Fraction of transactions containing the itemset.")
print("Confidence: Fraction of transactions with the antecedent that also contain the consequent.")
print("Lift: Strength of the rule compared with the consequent occurring independently.")
print("A lift greater than 1 indicates a positive association.")

## Conclusion

The Apriori algorithm identifies frequently occurring item combinations. Association rules describe relationships between items, while support, confidence, and lift help evaluate those relationships.

## Exercises / Assignments

**1.** Reduce minimum support to 0.2 and display the frequent itemsets.

In [ ]:
frequent_itemsets_02 = apriori(encoded_df, min_support=0.2, use_colnames=True)
display(frequent_itemsets_02)

**2.** Generate association rules using a higher confidence threshold of 0.7.

In [ ]:
rules_07 = association_rules(
    frequent_itemsets_for_rules,
    metric="confidence",
    min_threshold=0.7
)

if rules_07.empty:
    print("No rules meet the confidence threshold of 0.7.")
else:
    display(rules_07[['antecedents', 'consequents', 'support', 'confidence', 'lift']].round(3))

**3.** Add two transactions and compare the frequent itemsets.

In [ ]:
updated_transactions = transactions + [
    ['Milk', 'Bread'],
    ['Bread', 'Butter', 'Eggs']
]

updated_encoder = TransactionEncoder()
updated_array = updated_encoder.fit(updated_transactions).transform(updated_transactions)
updated_df = pd.DataFrame(updated_array, columns=updated_encoder.columns_)

updated_itemsets = apriori(updated_df, min_support=0.3, use_colnames=True)

print("Frequent itemsets after adding two transactions:")
display(updated_itemsets)

print("Original number of frequent itemsets:", len(frequent_itemsets_for_rules))
print("Updated number of frequent itemsets:", len(updated_itemsets))

**4.** Identify the rule with the highest lift value.

In [ ]:
if not rules.empty:
    highest_lift_rule = rules.loc[rules['lift'].idxmax()]
    print("Antecedent:", highest_lift_rule['antecedents'])
    print("Consequent:", highest_lift_rule['consequents'])
    print("Lift:", round(highest_lift_rule['lift'], 3))
else:
    print("No association rules are available.")

**5. Significance of the measures**

- **Support:** Shows how frequently an itemset occurs in all transactions.
- **Confidence:** Shows how often the consequent appears when the antecedent appears.
- **Lift:** Shows how much more often the items occur together than expected if they were independent. Lift above 1 indicates positive association.